# ⚖️ LexGPT — QLoRA Fine-tune Qwen2.5-7B trên Luật Việt Nam

## Tổng quan kỹ thuật
| Thành phần | Chi tiết |
|---|---|
| Base model | Qwen2.5-7B-Instruct (7 tỷ params) |
| Kỹ thuật | QLoRA = 4-bit quantization + LoRA (rank=16) |
| VRAM cần | ~11 GB → chạy được trên T4 (15GB) |
| Thời gian | ~60-90 phút trên T4 |
| Trainable params | ~25M / 7B = 0.36% |


In [ ]:
# ═══════════════════════════════════════════
# CELL 1: Kiểm tra GPU & môi trường
# ═══════════════════════════════════════════
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'✅ GPU    : {props.name}')
    print(f'   VRAM  : {props.total_memory/1e9:.1f} GB')
    print(f'   bf16  : {props.major >= 8}')
else:
    print('⚠️ Không có GPU! Vào Runtime → Change runtime type → T4 GPU')

print(f'PyTorch: {torch.__version__}')

In [ ]:
# ═══════════════════════════════════════════
# CELL 2: Cài thư viện cần thiết
# ═══════════════════════════════════════════
# transformers: load Qwen2.5
# peft        : LoRA adapters
# bitsandbytes: 4-bit quantization
# accelerate  : device management
!pip install transformers>=4.45.0 peft>=0.13.0 bitsandbytes>=0.43.0 accelerate>=0.33.0 -q
print('✅ Cài xong!')

In [ ]:
# ═══════════════════════════════════════════
# CELL 3: Upload file dữ liệu & code
# ═══════════════════════════════════════════
import os
from google.colab import files

os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

print('Bước 1: Upload 2 file CSV (blhs_2025_from_text.csv, giaothong.csv)')
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f'data/{name}')
    print(f'  ✅ {name}')

print('\nBước 2: Upload lexgpt_qwen.zip (project code)')
uploaded = files.upload()
!unzip -o lexgpt_qwen.zip -d . -q

import sys
sys.path.insert(0, 'lexgpt_qwen')
os.chdir('lexgpt_qwen')

# Copy CSV vào thư mục data của project
import shutil
for csv_file in ['blhs_2025_from_text.csv', 'giaothong.csv']:
    src = f'../data/{csv_file}'
    if os.path.exists(src):
        shutil.copy(src, f'data/{csv_file}')
        print(f'  📂 {csv_file} → data/')

print('\n✅ Setup xong!')

In [ ]:
# ═══════════════════════════════════════════
# CELL 4: HuggingFace Login (cần để download Qwen2.5)
# ═══════════════════════════════════════════
# Tạo tài khoản HuggingFace miễn phí tại: https://huggingface.co
# Vào Settings → Access Tokens → New token (Read)
from huggingface_hub import login
login()  # Nhập token khi được hỏi

In [ ]:
# ═══════════════════════════════════════════
# CELL 5: Bước 1 — Build Dataset
# ═══════════════════════════════════════════
from dataset.build_dataset import build_and_save

train_path, val_path = build_and_save(
    blhs_path   = 'data/blhs_2025_from_text.csv',
    gt_path     = 'data/giaothong.csv',
    output_path = 'data/legal_qa.jsonl',
)
print(f'Train: {train_path}')
print(f'Val  : {val_path}')

In [ ]:
# ═══════════════════════════════════════════
# CELL 6: Bước 2 — Load Qwen2.5-7B + QLoRA
# (Download ~14GB lần đầu — mất 10-15 phút)
# ═══════════════════════════════════════════
from model.load_qwen_qlora import load_qwen_qlora

tokenizer, model = load_qwen_qlora(
    model_name = 'Qwen/Qwen2.5-7B-Instruct',
    lora_r     = 16,
)

import torch
print(f'\nVRAM sau load: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# ═══════════════════════════════════════════
# CELL 7: Bước 3 — Tạo DataLoader
# ═══════════════════════════════════════════
from functools import partial
from torch.utils.data import DataLoader
from dataset.build_dataset import LegalChatDataset, collate_fn

train_ds = LegalChatDataset('data/legal_qa_train.jsonl', tokenizer, max_length=512)
val_ds   = LegalChatDataset('data/legal_qa_val.jsonl',   tokenizer, max_length=512)

pad_id = tokenizer.pad_token_id
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,
                          collate_fn=partial(collate_fn, pad_token_id=pad_id))
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False,
                          collate_fn=partial(collate_fn, pad_token_id=pad_id))

print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')

In [ ]:
# ═══════════════════════════════════════════
# CELL 8: Bước 4 — QLoRA Fine-tuning  ← CHÍNH
# Ước tính: ~60-90 phút trên T4
# ═══════════════════════════════════════════
from trainer.qlora_trainer import QLoRATrainer, QLoRAConfig

cfg = QLoRAConfig(
    num_epochs   = 3,
    batch_size   = 2,
    grad_accum   = 8,      # effective batch = 16
    lr           = 2e-4,
    max_length   = 512,
    log_every    = 5,
    eval_every   = 50,
    save_every   = 100,
)

trainer = QLoRATrainer(cfg)
model   = trainer.train(model, tokenizer, train_loader, val_loader)

print('\n🎉 Fine-tuning hoàn tất!')

In [ ]:
# ═══════════════════════════════════════════
# CELL 9: Test model
# ═══════════════════════════════════════════
from inference.chatbot import LexGPTQwen

bot = LexGPTQwen('checkpoints/best_model')

test_questions = [
    'Điều 260 BLHS 2025 quy định về tội gì?',
    'Uống rượu lái xe ô tô gây chết người bị phạt bao nhiêu năm tù?',
    'Vi phạm nồng độ cồn lần đầu tiên bị phạt bao nhiêu tiền?',
]

bot.evaluate_samples(test_questions)

In [ ]:
# ═══════════════════════════════════════════
# CELL 10: Download LoRA weights về máy
# (Chỉ vài MB — không phải 14GB!)
# ═══════════════════════════════════════════
import shutil
from google.colab import files

shutil.make_archive('lexgpt_lora_weights', 'zip', 'checkpoints/best_model')
files.download('lexgpt_lora_weights.zip')
print('✅ Download LoRA weights xong!')
print('   Dùng lại: load_with_adapter("checkpoints/best_model")')